In [1]:
from brainglobe_atlasapi.atlas_generation.mesh_utils import create_region_mesh
import json
from treelib import Tree
from pathlib import Path
import os
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from read_roi import read_roi_zip
import re
from matplotlib.colors import LogNorm
import matplotlib.colors as clrs
import skimage
from scipy.interpolate import interp1d
from matplotlib.path import Path as Mathpath
import pandas as pd
from natsort import natsorted
from collections import OrderedDict

In [2]:
# draw an roi.zip file in ImageJ on the right hand side of the atlas images
atlas_name = 'allen_mouse_io_10um_v1.0'
structures_json_path = f'C:/Users/Sam/.brainglobe/{atlas_name}/structures.json'
meshes_path = Path(f'C:/Users/Sam/.brainglobe/{atlas_name}/meshes')
debug = False

In [3]:
def get_slice_settings_for_area_ID(ID):
    if ID >= 6000 and ID <= 6011:
        substack_stepsize, substack_start_slice = 5, 1120
    elif ID == 6020:
        substack_stepsize, substack_start_slice = 5, 770
    else:
        raise Exception('ID not recognised')
    return substack_stepsize, substack_start_slice

## Extracted functions

In [4]:
def parent_in_tree(node):
    return structures_tree.parent(node.identifier)

In [5]:
def structure_ID_path(node):
    path_ID_array = node.data['structure_id_path']
    string = ''
    for ID in path_ID_array[:-1]:
        string = string + '/' + str(ID)
    return string

In [6]:
def generate_tree_from_json(json_file):
    with open(json_file, 'r') as file:
        json_data = json.load(file)
    
    tree = Tree()
    
    # Create a dictionary to store node IDs and their corresponding tree nodes
    node_dict = {}
    
    for item in json_data:
        node_id = item['id']
        parent_id = item['structure_id_path'][-2] if len(item['structure_id_path']) > 1 else None
        
        # Create a new node
        node = tree.create_node(
            tag=item['name'],
            identifier=node_id,
            parent=node_dict.get(parent_id),
            data={
                'acronym': item['acronym'],
                'structure_id_path': item['structure_id_path'],
                'rgb_triplet': item['rgb_triplet']
            }
        )
        
        # Store the node in the dictionary
        node_dict[node_id] = node
    
    return tree

In [7]:
def add_nodes_to_structures_csv(nodes, structures_tree):
    structures_path = f'C:/Users/Sam/.brainglobe/{atlas_name}/structures_original.csv'
    new_structures_path = f'C:/Users/Sam/.brainglobe/{atlas_name}/structures.csv'
    df = pd.read_csv(structures_path)
    
    rows_to_append = []
    for node in nodes:
        full_structure_ID_path = f'{structure_ID_path(node)}/{node.identifier}/'
        node_details_for_csv = {'acronym' : node.data['acronym'], 
                                'id' : node.identifier, 
                                'name' : node.tag, 
                                'structure_id_path' : full_structure_ID_path, 
                                'parent_structure_id' : parent_in_tree(node).identifier}
        rows_to_append.append(node_details_for_csv)
    
    new_data = pd.DataFrame(rows_to_append)
    extended_df = pd.concat([df, new_data], ignore_index=True)
    extended_df.to_csv(new_structures_path, index=False)

In [8]:
def open_image(path):
    assert os.path.isfile(path), f'Please provide image at {path}'
    image = imread(path)
    return image

In [9]:
def transparent_cmap(cmap_name):
    base_cmap = plt.get_cmap(cmap_name)
    colors = base_cmap(np.arange(base_cmap.N))
    # Set alpha channel from 0 (transparent) at the bottom to 1 at the top
    colors[:, -1] = np.linspace(0, 1, base_cmap.N)
    return clrs.LinearSegmentedColormap.from_list(f'transparent_{cmap_name}', colors)

In [10]:
def show_annotation(single_slice):
    f, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20,25))
    clipped_annotated_volume = np.clip(annotated_volume, None, 11_000)
    ax1.imshow(np.mean(clipped_annotated_volume, axis=0))
    ax1.set_title('Whole annotated volume (mean proj.)')
    clipped_annotated_volume = np.clip(annotated_volume, None, 2_000)
    ax2.imshow(np.mean(clipped_annotated_volume[1130:1275,:,:], axis=0))
    ax2.set_title('Clipped + sliced to IO sections (mean proj.)')
    ax3.imshow(annotated_volume[single_slice,:,:], vmax=2000)
    ax3.set_title('Single slice')

In [11]:
def setup_ax_background(ax, axis, volume, parent_region_of_interest):
    data = np.max((volume == parent_region_of_interest), axis=axis)
    if axis == 2:
        data = data.T
    ax.imshow(data, cmap=transparent_cmap('Reds'))
    tick_positions = np.arange(0, volume.shape[0], 100)
    ax.set_xticks(tick_positions)
    ax.set_yticks(tick_positions)
    ax.grid(which='both', color='gray', linestyle='-', linewidth=0.5)

In [12]:
def show_volume(volume, annotated_volume, parent_region_of_interest):
    f, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20,25))
    setup_ax_background(ax1, 0, annotated_volume, parent_region_of_interest)
    setup_ax_background(ax2, 2, annotated_volume, parent_region_of_interest)
    setup_ax_background(ax3, 1, annotated_volume, parent_region_of_interest)
    ax1.imshow(np.max(volume, axis=0), cmap=transparent_cmap('plasma'))
    ax2.imshow(np.max(volume, axis=2).T, cmap=transparent_cmap('plasma'))
    ax3.imshow(np.max(volume, axis=1), cmap=transparent_cmap('plasma'))
    ax3.set_title('Horizontal'); ax1.set_title('Coronal'); ax2.set_title('Sagittal');

In [13]:
def set_z_idxs_for_ROIs(ROIs, substack_stepsize, substack_start_slice):
    for key, roi in ROIs.items():
        # Extract slice number from key, e.g. "dPO_6" -> 6
        z_idx = ((int(key.split('_')[1]) - 1) * substack_stepsize) + substack_start_slice
        #z_positions_for_rois.append(z_idx)
        ROIs[key]['z'] = z_idx
    if debug:
        print([key for key in ROIs.keys()])
        print([roi['z'] for roi in ROIs.values()])

In [14]:
def apply_region_to_mask(ROIs, mask_3d):
    for key, roi in ROIs.items():
        x_coords = np.array(roi['x'])
        y_coords = np.array(roi['y'])
        
        # Create polygon path for this slice
        poly_path = Mathpath(np.column_stack((x_coords, y_coords)))
        
        # Create a grid of points in 2D for the slice
        # For efficiency, limit to bounding box of polygon
        xmin = int(np.floor(x_coords.min()))
        xmax = int(np.ceil(x_coords.max()))
        ymin = int(np.floor(y_coords.min()))
        ymax = int(np.ceil(y_coords.max()))
        
        x_grid, y_grid = np.meshgrid(np.arange(xmin, xmax + 1), np.arange(ymin, ymax + 1))
        points = np.vstack((x_grid.flatten(), y_grid.flatten())).T
        
        # Check which points are inside the polygon
        inside = poly_path.contains_points(points)
        inside_points = points[inside]
        
        # Set True in mask for these points in appropriate z slice
        mask_3d[roi['z'], inside_points[:, 1], inside_points[:, 0]] = True

In [15]:
def interpolate_along_mask(ROIs, mask_3d):
    z_positions_for_rois = [roi['z'] for roi in ROIs.values()]
    
    # mask_3d boolean (Z, Y, X)
    mask_selected = mask_3d[z_positions_for_rois, :, :]
    mask_float = mask_selected.astype(float)
    Z, Y, X = mask_selected.shape
    
    # Define original z positions corresponding to mask slices
    z_known = np.array(z_positions_for_rois)  # length Z
    
    # Reshape mask to (Z, Y*X)
    mask_2d = mask_float.reshape(Z, -1)
    
    # Define new finer z positions for interpolation
    at_real_resolution = z_known.max() - z_known.min()
    z_new = np.linspace(z_known.min(), z_known.max(), at_real_resolution)
    
    # Prepare output array
    mask_interp_2d = np.empty((len(z_new), Y*X), dtype=float)
    
    # Loop over each flattened pixel coordinate for interpolation along z
    for i in range(Y*X):
        mask_interp_2d[:, i] = np.interp(z_new, z_known, mask_2d[:, i], left=0, right=0)
    
    # Reshape back to 3D
    mask_interp_float = mask_interp_2d.reshape(len(z_new), Y, X)
    
    # Threshold to boolean mask
    mask_interp_bool = mask_interp_float >= 0.5
    
    original_z_min, original_z_max = np.min(z_positions_for_rois), np.max(z_positions_for_rois)
    mask_3d[original_z_min : original_z_max] = mask_interp_bool

In [16]:
def reflect_along_midline(mask_3d):
    midpoint_x = int(mask_3d.shape[2]/2)
    right_hand_substack = mask_3d[:, :, midpoint_x:]
    mask_3d[:, :, :midpoint_x] = right_hand_substack[:, :, ::-1]

In [17]:
def apply_imageJ_ROIs_to_annotation_image(value, ROIs, annotated_volume, substack_stepsize, substack_start_slice, parent_region_of_interest):
    set_z_idxs_for_ROIs(ROIs, substack_stepsize, substack_start_slice)
    mask_3d = np.zeros(annotated_volume.shape, dtype=bool) # Initialize 3D mask with False (shape: z, y, x)
    
    apply_region_to_mask(ROIs, mask_3d)
    interpolate_along_mask(ROIs, mask_3d)
    reflect_along_midline(mask_3d)
    if debug:
        show_volume(mask_3d, annotated_volume, parent_region_of_interest)
    annotated_volume[mask_3d] = value

In [18]:
def get_subset_of_nodes_in_annotated_volume(nodes):
    nodes_subset = []
    labels_in_annotated_volume = get_labels_in_volume(annotated_volume)
    for node in nodes:
        if node.identifier in labels_in_annotated_volume:
            nodes_subset.append(node)
    print(f'Areas {[node.identifier for node in nodes_subset]} found that exist in the annotated volume.')
    return nodes_subset

In [19]:
def get_labels_in_volume(volume):
    return list(np.unique(volume))

In [20]:
def get_subset_of_regions_in_structures_json_without_meshes(structures_tree):
    mesh_file_names = [f.name for f in meshes_path.iterdir() if f.is_file()]
    identifiers_with_existing_meshes = [int(re.search(r'\d+', fname).group()) for fname in mesh_file_names] + [545] # Allen has region 545 but assigns it no volume

    nodes_without_mesh = []
    for node in structures_tree.all_nodes():
        if node.identifier not in identifiers_with_existing_meshes:
            nodes_without_mesh.append(node)
    print(f"Identifiers {[node.identifier for node in nodes_without_mesh]} do not already have a mesh.")
    return nodes_without_mesh

In [21]:
def sort_ordered_dict_naturally(od):
    sorted_keys = natsorted(od.keys())
    return OrderedDict((k, od[k]) for k in sorted_keys)

## Main code

In [22]:
structures_tree = generate_tree_from_json(structures_json_path) #tree.show()

In [23]:
nodes_without_mesh = get_subset_of_regions_in_structures_json_without_meshes(structures_tree)

Identifiers [6000, 6002, 6003, 6004, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6020] do not already have a mesh.


In [24]:
add_nodes_to_structures_csv(nodes_without_mesh, structures_tree)

In [25]:
annotated_volume = open_image(f'C:/Users/Sam/.brainglobe/{atlas_name}/annotation_original.tiff')
if debug:
    show_annotation(single_slice=1170)

In [26]:
for node in nodes_without_mesh:
    ID = node.identifier
    acronym = node.data['acronym']
    print(ID, acronym)
    try:
        ROIs = read_roi_zip(f'{acronym}.zip')
    except Exception as e:
        print(e)
    else:
        ROIs = sort_ordered_dict_naturally(ROIs)
        substack_stepsize, substack_start_slice = get_slice_settings_for_area_ID(ID)
        parent_region_of_interest = parent_in_tree(node)
        apply_imageJ_ROIs_to_annotation_image(ID, ROIs, annotated_volume, substack_stepsize, substack_start_slice, parent_region_of_interest)

6000 DC
6002 Beta-VLO
6003 dDAO
6004 dPO
6005 vDAO
6006 DM-vPO
6007 rMAO
6008 DMCC
6009 cMAO-a
6010 cMAO-b
6011 cMAO-c
6020 MDJ


In [27]:
if debug:
    show_annotation(single_slice=1170)
skimage.io.imsave(f'C:/Users/Sam/.brainglobe/{atlas_name}/annotation.tiff', annotated_volume)

In [28]:
nodes_to_gen_mesh_for = get_subset_of_nodes_in_annotated_volume(nodes_without_mesh) # regions without a mesh that are in the volume
labels_in_annotated_volume = get_labels_in_volume(annotated_volume)
ROOT_ID = 997
for node in nodes_to_gen_mesh_for:
    create_region_mesh([meshes_path, node, structures_tree, labels_in_annotated_volume, annotated_volume, ROOT_ID, 8, 0.6, False])
# meshes_dir_path: pathlib Path object with folder where meshes are saved
# tree: treelib.Tree with hierarchical structures information
# node: tree's node corresponding to the region who's mesh is being created
# labels: list of unique label annotations in annotated volume
# annotated_volume: 3d numpy array with annotaed volume
# ROOT_ID: int, id of root structure (mesh creation is a bit more refined for that)

Areas [6000, 6002, 6003, 6004, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6020] found that exist in the annotated volume.


2025-09-05 14:47:28.546 | DEBUG    | brainglobe_atlasapi.atlas_generation.mesh_utils:create_region_mesh:195 - Creating mesh for region 6000
2025-09-05 14:49:55.638 | DEBUG    | brainglobe_atlasapi.atlas_generation.mesh_utils:create_region_mesh:195 - Creating mesh for region 6002
2025-09-05 14:52:18.625 | DEBUG    | brainglobe_atlasapi.atlas_generation.mesh_utils:create_region_mesh:195 - Creating mesh for region 6003
2025-09-05 14:54:46.830 | DEBUG    | brainglobe_atlasapi.atlas_generation.mesh_utils:create_region_mesh:195 - Creating mesh for region 6004
2025-09-05 14:57:22.299 | DEBUG    | brainglobe_atlasapi.atlas_generation.mesh_utils:create_region_mesh:195 - Creating mesh for region 6005
2025-09-05 14:59:53.370 | DEBUG    | brainglobe_atlasapi.atlas_generation.mesh_utils:create_region_mesh:195 - Creating mesh for region 6006
2025-09-05 15:02:15.540 | DEBUG    | brainglobe_atlasapi.atlas_generation.mesh_utils:create_region_mesh:195 - Creating mesh for region 6007
2025-09-05 15:04:37.